# Scenario B — DB → Elasticsearch (Bulk Update)

Records found **in PostgreSQL but missing from Elasticsearch**.  
Run each step in order. Review and download data at each stage before moving on.

| Step | Action |
|------|--------|
| 1 | Load session from `1_compare.ipynb` + enter credentials |
| 2 | View DB-not-in-ES records |
| 3 | Filter already-deleted records |
| 4 | Find & remove test data (marks deleted in DB via SQL) |
| 5 | Fetch full task objects from HCM API |
| 6 | Run Bulk Update |
| — | Retry failed records (if needed) |

In [ ]:
import json, os, sys, warnings
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML
warnings.filterwarnings('ignore')

repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline import filter as _flt
from pipeline import ingest as _ing

with open('campaign_config.json') as f:
    _raw = json.load(f)
_GLOBALS  = _raw.get('_globals', {})
CAMPAIGNS = {k: v for k, v in _raw.items() if k != '_globals'}

def get_cfg(key):
    return {**_GLOBALS, **CAMPAIGNS[key]}

def _dl(path, label='Download'):
    if path and os.path.exists(str(path)):
        display(FileLink(str(path), result_html_prefix=f'\u2b07  {label}: '))

# Shared state across steps
SESSION      = {}
CFG          = {}
DB_CONFIG    = {}
DB_RAW_DF    = pd.DataFrame()   # loaded from compare output
DB_ACTIVE_DF = pd.DataFrame()   # after isdeleted filter
DB_CLEAN_DF  = pd.DataFrame()   # after test data removal
TASKS_FROM_API = []

print('Setup complete. Run the cells below in order.')

In [ ]:
_W = {'description_width': '110px'}

w_campaign = widgets.Dropdown(
    options=[(v['label'], k) for k, v in CAMPAIGNS.items()],
    description='Campaign:', style=_W, layout=widgets.Layout(width='340px')
)
w_outdir = widgets.Text(value='output/', description='Output dir:', style=_W, layout=widgets.Layout(width='340px'))

btn_load = widgets.Button(description='Load', button_style='info', icon='folder-open',
                          layout=widgets.Layout(width='160px'))
out_load = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _load(b):
    global SESSION, CFG, DB_CONFIG
    with out_load:
        out_load.clear_output()
        try:
            key = w_campaign.value
            CFG = get_cfg(key)
            DB_CONFIG = {
                'host': CFG['db_host'], 'port': CFG['db_port'], 'database': CFG['db_name'],
                'user': CFG['db_user'], 'password': CFG['db_pass'],
                'sslmode': 'require', 'connect_timeout': 30,
            }
            base = w_outdir.value.strip().rstrip('/')
            output_dir = os.path.join(base, key) if os.path.isdir(os.path.join(base, key)) else base
            if not os.path.isdir(output_dir):
                print(f'ERROR: Directory not found: {output_dir}')
                print('Run 1_compare.ipynb first.')
                return
            marker_files = sorted(f for f in os.listdir(output_dir) if f.startswith('DB_not_in_Elastic_'))
            if not marker_files:
                print(f'ERROR: No compare output found in {output_dir}')
                print('Run 1_compare.ipynb first.')
                return
            from datetime import datetime
            SESSION = {
                'campaign_key': key,
                'output_dir':   output_dir,
                'ts':           datetime.now().strftime('%Y%m%d_%H%M%S'),
            }
            display(HTML(
                f"<div style='font-family:monospace;background:#f3fff3;padding:10px;border-radius:4px;border:1px solid #bdb'>"
                f"✓ Ready<br>"
                f"Campaign  : <b>{key}</b> ({CFG['label']})<br>"
                f"Output dir: {output_dir}<br>"
                f"Latest run: {marker_files[-1]}<br>"
                f"DB: {CFG['db_host']} / {CFG['db_name']} as {CFG['db_user']}"
                f"</div>"
            ))
        except Exception:
            import traceback; traceback.print_exc()

btn_load.on_click(_load)
display(widgets.HTML('<h3>Setup</h3>'))
display(w_campaign, w_outdir, btn_load, out_load)

In [ ]:
btn_s1 = widgets.Button(description='Step 1 — Load DB Data', button_style='primary', icon='eye',
                        layout=widgets.Layout(width='250px'))
out_s1 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step1(b):
    global DB_RAW_DF
    with out_s1:
        out_s1.clear_output()
        try:
            output_dir = SESSION['output_dir']
            all_files  = os.listdir(output_dir)
            detail_files = sorted(f for f in all_files if f.startswith('DB_details_not_in_elastic'))
            id_files     = sorted(f for f in all_files if f.startswith('DB_not_in_Elastic_'))
            if detail_files:
                path = os.path.join(output_dir, detail_files[-1])
            elif id_files:
                path = os.path.join(output_dir, id_files[-1])
            else:
                print('ERROR: No DB compare output file found. Run 1_compare.ipynb first.')
                return
            DB_RAW_DF = pd.read_csv(path)
            print(f'Loaded {len(DB_RAW_DF):,} DB records not found in ES  ({os.path.basename(path)})')
            if len(DB_RAW_DF) == 0:
                print('No records missing from ES — nothing to do for Scenario B.')
                return
            print(f'Columns: {list(DB_RAW_DF.columns)}')
            display(DB_RAW_DF.head(20))
            _dl(path, 'DB-not-in-ES details')
        except Exception:
            import traceback; traceback.print_exc()

btn_s1.on_click(_step1)
display(widgets.HTML('<h3>Step 1 — View DB Records Not in Elasticsearch</h3>'))
display(btn_s1, out_s1)

In [ ]:
btn_s2 = widgets.Button(description='Step 2 — Filter isDeleted', button_style='primary', icon='filter',
                        layout=widgets.Layout(width='260px'))
out_s2 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step2(b):
    global DB_ACTIVE_DF
    with out_s2:
        out_s2.clear_output()
        try:
            col = next((c for c in DB_RAW_DF.columns if c.lower() == 'isdeleted'), None)
            if col:
                already_del  = DB_RAW_DF[DB_RAW_DF[col] == True]
                DB_ACTIVE_DF = DB_RAW_DF[DB_RAW_DF[col] != True].copy()
                print(f'Already isdeleted  : {len(already_del):,}')
                print(f'Active (to process): {len(DB_ACTIVE_DF):,}')
                if len(already_del):
                    p = os.path.join(SESSION['output_dir'], f"removed_isdeleted_{SESSION['ts']}.xlsx")
                    already_del.to_excel(p, index=False)
                    _dl(p, 'Already-deleted records')
            else:
                print('No isdeleted column found — treating all records as active')
                DB_ACTIVE_DF = DB_RAW_DF.copy()
            display(DB_ACTIVE_DF.head(20))
        except Exception:
            import traceback; traceback.print_exc()

btn_s2.on_click(_step2)
display(widgets.HTML('<h3>Step 2 — Filter Already-Deleted Records</h3>'))
display(btn_s2, out_s2)

In [ ]:
btn_s3a = widgets.Button(description='3a — Find Test Data', button_style='warning', icon='search',
                         layout=widgets.Layout(width='220px'))
btn_s3b = widgets.Button(description='3b — Mark Deleted in DB (SQL)', button_style='danger', icon='trash',
                         layout=widgets.Layout(width='270px'))
btn_s3b.disabled = True
out_s3 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))
TEST_IDS = []

def _step3a(b):
    global DB_CLEAN_DF, TEST_IDS
    with out_s3:
        out_s3.clear_output()
        try:
            tuc      = CFG.get('test_user_config', {})
            patterns = tuc.get('db_test_name_patterns', [])
            ind_tbl  = tuc.get('db_individual_table', '')
            name_col = tuc.get('db_individual_name_col', 'username')

            DB_CLEAN_DF, test_df = _flt.filter_test_data_db(
                df                = DB_ACTIVE_DF,
                db_config         = DB_CONFIG,
                individual_table  = ind_tbl,
                name_col          = name_col,
                test_name_patterns= patterns,
                createdby_col     = 'createdby',
            )

            print(f'Test records found  : {len(test_df):,}')
            print(f'Clean records remain: {len(DB_CLEAN_DF):,}')

            if len(test_df):
                id_col = CFG['task']['db_id_column']
                TEST_IDS = test_df[id_col].dropna().tolist() if id_col in test_df.columns else []
                display(HTML('<b>Test data preview (first 20 rows):</b>'))
                display(test_df.head(20))
                p = os.path.join(SESSION['output_dir'], f"test_data_db_{SESSION['ts']}.xlsx")
                test_df.to_excel(p, index=False)
                _dl(p, 'Test data')
                btn_s3b.disabled = False
                display(HTML("<br><span style='color:orange'>\u26a0 Review above. Click <b>3b</b> to mark these as deleted in DB via SQL UPDATE.</span>"))
            else:
                print('No test data found. Proceed to Step 4.')
        except Exception:
            import traceback; traceback.print_exc()

def _step3b(b):
    btn_s3b.disabled = True
    with out_s3:
        try:
            if not TEST_IDS:
                print('No test IDs to mark.')
                return
            task_cfg = CFG['task']
            print(f'Running SQL UPDATE on {task_cfg["db_table"]} for {len(TEST_IDS):,} records...')
            updated = _flt.mark_deleted_in_db(
                DB_CONFIG, task_cfg['db_table'], task_cfg['db_id_column'], TEST_IDS
            )
            display(HTML(f"<span style='color:green'>\u2713 {updated:,} rows marked isdeleted=true in DB</span>"))
        except Exception:
            import traceback; traceback.print_exc()
            btn_s3b.disabled = False

btn_s3a.on_click(_step3a)
btn_s3b.on_click(_step3b)
display(widgets.HTML('<h3>Step 3 — Identify &amp; Remove Test Data</h3>'))
display(widgets.HTML("<small>Test records are identified by looking up <code>createdby</code> UUID in the <code>individual</code> table and checking for patterns like 'test', 'demo'.</small>"))
display(widgets.HBox([btn_s3a, btn_s3b]), out_s3)

In [ ]:
btn_s4 = widgets.Button(description='Step 4 — Fetch from HCM API', button_style='primary', icon='cloud-download',
                        layout=widgets.Layout(width='280px'))
out_s4 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step4(b):
    global TASKS_FROM_API
    btn_s4.disabled = True
    with out_s4:
        out_s4.clear_output()
        try:
            df = DB_CLEAN_DF if len(DB_CLEAN_DF) else DB_ACTIVE_DF if len(DB_ACTIVE_DF) else DB_RAW_DF
            if not len(df):
                print('ERROR: No data loaded. Run Steps 1 and 2 first.')
                return

            # Detect ID column — details file has exact name, ID-list file has _missing_in_elastic suffix
            id_col = CFG['task']['db_id_column']
            col = id_col if id_col in df.columns else \
                  next((c for c in df.columns if 'clientreferenceid' in c.lower()), None)
            if not col:
                print(f'ERROR: Cannot find ID column. Columns available: {list(df.columns)}')
                return

            ids = df[col].dropna().astype(str).tolist()
            print(f'Fetching {len(ids):,} tasks from HCM API...')
            print(f'  {CFG["api_base"] + CFG["api_paths"]["task_search"]}\n')

            TASKS_FROM_API = _ing.fetch_tasks_from_api(
                api_base             = CFG['api_base'],
                tenant_id            = CFG['tenant_id'],
                auth_token           = CFG['auth_token'],
                client_reference_ids = ids,
            )

            print(f'\nFetched {len(TASKS_FROM_API):,} task objects from API')
            not_fetched = len(ids) - len(TASKS_FROM_API)
            if not_fetched:
                print(f'Not returned by API: {not_fetched:,} (may already exist in ES or be stale)')

            ap = os.path.join(SESSION['output_dir'], f"api_tasks_{SESSION['ts']}.json")
            with open(ap, 'w') as f:
                json.dump(TASKS_FROM_API, f, indent=2)
            _dl(ap, 'Fetched tasks JSON')

            display(HTML('<br><b>Preview (first 2 records):</b>'))
            display(HTML(f"<pre style='background:#f9f9f9;padding:8px;max-height:300px;overflow:auto'>{json.dumps(TASKS_FROM_API[:2], indent=2)}</pre>"))
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_s4.disabled = False

btn_s4.on_click(_step4)
display(widgets.HTML('<h3>Step 4 — Fetch Task Objects from HCM API</h3>'))
display(btn_s4, out_s4)

In [ ]:
btn_s5 = widgets.Button(description='Step 5 — Run Bulk Update', button_style='danger', icon='upload',
                        layout=widgets.Layout(width='260px'))
out_s5 = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _step5(b):
    btn_s5.disabled = True
    with out_s5:
        out_s5.clear_output()
        try:
            if not TASKS_FROM_API:
                print('ERROR: No tasks fetched. Run Step 4 first.')
                return
            api_url     = CFG['api_base'] + CFG['api_paths']['task_update']
            failed_path = os.path.join(SESSION['output_dir'], f"failed_update_{SESSION['ts']}.json")

            print(f'Sending {len(TASKS_FROM_API):,} tasks to:')
            print(f'  {api_url}\n')

            result = _ing.bulk_ingest(
                api_url     = api_url,
                auth_token  = CFG['auth_token'],
                tasks       = TASKS_FROM_API,
                failed_path = failed_path,
            )

            display(HTML(
                f"<div style='font-family:monospace;margin-top:8px'>"
                f"<span style='color:green'>\u2713 Success: {result['success']:,}</span><br>"
                f"<span style='color:{'red' if result['failed'] else 'green'}'>\u2715 Failed : {result['failed']:,}</span>"
                f"</div>"
            ))
            if result['failed']:
                _dl(failed_path, 'Failed records — use Retry cell below')
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_s5.disabled = False

btn_s5.on_click(_step5)
display(widgets.HTML('<h3>Step 5 — Bulk Update (DB &#8594; Elasticsearch)</h3>'))
display(widgets.HTML(
    "<div style='background:#fff3f3;padding:8px 12px;border-radius:4px;border:1px solid #fcc;margin-bottom:6px'>"
    "\u26a0 This writes data to the HCM API. Confirm Steps 1&#8211;4 are correct before clicking."
    "</div>"
))
display(btn_s5, out_s5)

In [ ]:
btn_retry = widgets.Button(description='Retry Failed Records', button_style='warning', icon='refresh',
                           layout=widgets.Layout(width='230px'))
out_retry = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='6px 0'))

def _retry(b):
    btn_retry.disabled = True
    with out_retry:
        out_retry.clear_output()
        try:
            failed_path = os.path.join(SESSION['output_dir'], f"failed_update_{SESSION['ts']}.json")
            if not os.path.exists(failed_path):
                print('No failed_update file found.')
                return
            api_url = CFG['api_base'] + CFG['api_paths']['task_update']
            result  = _ing.retry_failed(failed_path, api_url, CFG['auth_token'])
            print(f"Retry — Success: {result['success']:,} | Failed: {result['failed']:,}")
            if result['failed']:
                _dl(failed_path, 'Still-failed records')
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            btn_retry.disabled = False

btn_retry.on_click(_retry)
display(widgets.HTML('<h3>Retry Failed Records (optional)</h3>'))
display(btn_retry, out_retry)